#embedding products

In [1]:
import pandas as pd
import requests
import numpy as np
import zlib
import time
from PIL import Image
from io import BytesIO
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer

# --- CONFIGURATION ---
client = QdrantClient(path="qdrant_local_data")
model = SentenceTransformer('clip-ViT-B-32')

PRODUCTS_COLLECTION = "products_db"
USERS_COLLECTION = "users_db"

# --- 1. UNIVERSAL ID CREATOR (Fixes the matching issue) ---
def get_universal_id(val):
    """
    Converts any ID (PROD-001, 123.0, etc.) into a consistent Integer.
    """
    s = str(val).strip()
    s = s.replace("PROD-", "").replace("USER-", "").replace("REV-", "")
    try:
        return int(float(s))
    except ValueError:
        return zlib.adler32(s.encode('utf-8')) & 0xffffffff

# --- 2. EMBED PRODUCTS (FULL DOWNLOAD) ---
def embed_products():
    print(f"\n📦 PART 1: Processing Products (Downloading ALL images)...")
    
    df_products = pd.read_csv("products.csv") 
    df_reviews = pd.read_csv("reviews.csv")

    # A. Clean IDs
    df_products['uni_id'] = df_products['id'].apply(get_universal_id)
    df_reviews['uni_pid'] = df_reviews['product_id'].apply(get_universal_id)

    # B. Concatenate Reviews into Product Data
    # We take the top 5 reviews for every product and join them into one text block
    reviews_grouped = df_reviews.groupby('uni_pid')['body'].apply(
        lambda x: " ".join(x.astype(str).tolist()[:5])
    ).reset_index()
    
    # Merge them: Now each product has its description AND reviews
    df_final = pd.merge(df_products, reviews_grouped, left_on='uni_id', right_on='uni_pid', how='left')
    
    total_items = len(df_final)
    print(f"   Target: {total_items} unique products found.")

    # C. Reset Qdrant
    if client.collection_exists(PRODUCTS_COLLECTION):
        client.delete_collection(PRODUCTS_COLLECTION)
    
    client.create_collection(
        collection_name=PRODUCTS_COLLECTION,
        vectors_config={
            "text": VectorParams(size=512, distance=Distance.COSINE),
            "image": VectorParams(size=512, distance=Distance.COSINE),
        }
    )

    points = []
    
    # D. Processing Loop
    print("   Starting download & embedding process...")
    start_time = time.time()
    
    for i, row in df_final.iterrows():
        
        # --- PROGRESS BAR ---
        if i % 5 == 0:
            elapsed = time.time() - start_time
            print(f"   Downloading & Processing: {i}/{total_items} ({(i/total_items)*100:.1f}%)", end="\r")

        # 1. TEXT ENCODING (Concatenated Data)
        review_text = str(row['body']) if pd.notnull(row['body']) else ""
        # Combine: Name + Category + Description + Reviews
        full_text_context = f"{row['name']} | {row['category']} | {row['description']} | Reviews: {review_text}"
        text_vec = model.encode(full_text_context).tolist()
        
        # 2. IMAGE DOWNLOAD & ENCODING
        image_vec = None
        if pd.notnull(row['image_url']):
            try:
                # 3-second timeout gives time to load, but prevents infinite hanging
                response = requests.get(row['image_url'], timeout=3)
                if response.status_code == 200:
                    img_obj = Image.open(BytesIO(response.content))
                    image_vec = model.encode(img_obj).tolist()
            except Exception:
                # If image is broken, we skip ONLY the image part, not the product
                pass 

        # 3. CONSTRUCT POINT
        vectors_dict = {"text": text_vec}
        if image_vec:
            vectors_dict["image"] = image_vec

        points.append(PointStruct(
            id=int(row['uni_id']),
            vector=vectors_dict,
            payload={
                "name": row['name'],
                "category": row['category'],
                "price": float(row['price']) if pd.notnull(row['price']) else 0.0
            }
        ))
        
        # Upload in batches of 50
        if len(points) >= 50:
            client.upsert(PRODUCTS_COLLECTION, points)
            points = []

    # Final batch upload
    if points:
        client.upsert(PRODUCTS_COLLECTION, points)
    
    print(f"\n✅ Products done! Processed {total_items} items.")

# --- EXECUTION ---
if __name__ == "__main__":
    embed_products()
    print("\n🎉 DONE! Full Database (Images + Text + Reviews) Created.")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.



📦 PART 1: Processing Products (Downloading ALL images)...
   Target: 2000 unique products found.
   Starting download & embedding process...
✅ Products done! Processed 2000 items.

👥 PART 2: Processing Users...
   Calculating preferences for 113 users...
   User 100...
⚠️ No users uploaded. Check errors above.

🎉 DONE! Full Database (Images + Text + Reviews) Created.


# implementing the user search

In [1]:
import pandas as pd
import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, Range
from sentence_transformers import SentenceTransformer
from PIL import Image
from groq import Groq

# --- CONFIGURATION ---
client = QdrantClient(path="qdrant_local_data") 
model = SentenceTransformer('clip-ViT-B-32')
COLLECTION_NAME = "products_db"

class PersonalizedSearchEngine:
    def __init__(self, reviews_df, behaviors_df, users_df, groq_api_key):
        self.reviews_df = reviews_df
        self.behaviors_df = behaviors_df
        self.users_df = users_df
        self.groq_client = Groq(api_key=groq_api_key)
        
        # Cleanup column names
        if 'id;user_id;email;first_name...' in self.users_df.columns: 
             self.users_df = self._split_user_columns(self.users_df)

    def _split_user_columns(self, df):
        col_name = df.columns[0]
        if ';' in col_name:
            split_data = df[col_name].str.split(';', expand=True)
            split_data.columns = col_name.split(';')
            return split_data
        return df

    def get_user_profile(self, user_id):
        try:
            user = self.users_df[self.users_df['user_id'] == user_id].iloc[0]
            return {
                'user_id': user_id,
                'monthly_limit': float(user.get('monthly_limit', 10000)) if pd.notna(user.get('monthly_limit')) else 10000.0,
                'preferred_sectors': str(user.get('preferred_sectors', '')),
            }
        except:
            return {'user_id': user_id, 'monthly_limit': 10000.0}

    def get_user_behavior_insights(self, user_id):
        recent = self.behaviors_df[self.behaviors_df['user_id'] == user_id]
        return {
            'viewed_products': recent[recent['event_type'] == 'view']['product_id'].tolist(),
            'favorite_categories': recent['product_category'].value_counts().to_dict(),
        }

    # --- 🟢 UPDATED EXPLANATION LOGIC ---
    def _generate_explanation(self, product_name, price, query_text):
        """
        Focuses STRICTLY on why the product matches the SEARCH QUERY.
        Ignores user hobbies if they are irrelevant to the current task.
        """
        try:
            # New Prompt: Focus on the Query + Value for Money
            prompt = (
                f"I am searching for '{query_text}'. "
                f"Explain why the product '{product_name}' (Price: ${price}) is a great choice for this specific search. "
                "Do not mention unrelated user hobbies. Focus on specs, value, or relevance. "
                "Keep it to 1 persuasive sentence."
            )
            
            chat_completion = self.groq_client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model="llama-3.3-70b-versatile",
                temperature=0.5,
                max_tokens=60
            )
            return chat_completion.choices[0].message.content.strip()
        except Exception:
            return "Great match for your search criteria."

    def personalized_search(self, user_id, query, search_type="text", limit=5, max_budget=None):
        print(f"\n🔎 SEARCHING ({search_type}): '{query}' for User {user_id}...")
        
        profile = self.get_user_profile(user_id)
        behavior = self.get_user_behavior_insights(user_id)
        
        final_budget = max_budget if max_budget else profile['monthly_limit']
        print(f"   💰 Max Budget Set To: ${final_budget}")

        # 1. Encode Query
        try:
            if search_type == "image":
                img_obj = Image.open(query)
                query_vector = model.encode(img_obj).tolist()
                query_context = "this uploaded image" # Context for AI explanation
            else:
                query_vector = model.encode(query).tolist()
                query_context = query # Context for AI explanation
        except Exception as e:
            print(f"❌ Error encoding query: {e}")
            return []
        
        price_filter = Filter(
            must=[FieldCondition(key="price", range=Range(lte=final_budget))]
        )

        # 2. Vector Search
        try:
            hits = client.query_points(
                collection_name=COLLECTION_NAME,
                query=query_vector,
                using="text",
                query_filter=price_filter,
                limit=15, 
                with_payload=True
            )
        except Exception:
            hits = client.query_points(
                collection_name=COLLECTION_NAME,
                query=query_vector,
                using="image", 
                query_filter=price_filter,
                limit=15, 
                with_payload=True
            )

        # 3. Score Boosting (Personalization runs SILENTLY in the background)
        results = []
        for point in hits.points:
            p_data = point.payload
            score = point.score
            
            # We still boost the score based on preferences, 
            # but we don't force the AI to talk about it unless it's relevant.
            cat = p_data.get('category', '')
            if cat in behavior['favorite_categories']:
                score *= 1.1 
            
            if p_data.get('sector', '') in profile['preferred_sectors']:
                score *= 1.15 

            results.append({
                "id": point.id,
                "name": p_data.get('name', 'Unknown Product'),
                "price": p_data.get('price', 0),
                "category": cat,
                "score": score
            })

        results.sort(key=lambda x: x['score'], reverse=True)
        final_results = results[:limit]

        # 4. Generate Explanations (Now Context-Aware)
        print("   🤖 Generating Smart AI explanations...")
        for res in final_results:
            # We pass the QUERY context, not the User Profile
            res['explanation'] = self._generate_explanation(
                res['name'], 
                res['price'], 
                query_context 
            )

        return final_results

# --- DISPLAY FUNCTION ---
def display_personalized_results(results):
    if not results:
        print("\n❌ No results found matching your criteria.")
        return

    print(f"\n✨ Top {len(results)} Personalized Recommendations:")
    print("=" * 60)

    for i, product in enumerate(results, 1):
        print(f"\n{i}. 📦 {product['name']}")
        print(f"   💰 Price: ${product['price']:.2f}")
        print(f"   📂 Category: {product['category']}")
        print(f"   ⭐ Relevance Score: {product['score']:.4f}")
        if 'explanation' in product:
            print(f"   💡 Why this fits: \"{product['explanation']}\"")
        print("-" * 60)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [2]:
# --- 1. Load Your REAL Data ---
print("📂 Loading your data files...")

try:
    df_users = pd.read_csv("users.csv", sep=";") 
    df_behaviors = pd.read_csv("behaviors.csv") 
    df_reviews = pd.read_csv("reviews.csv")    
    print("✅ Data loaded successfully!")
except Exception as e:
    print(f"❌ Error loading files: {e}")

# --- 2. Initialize the Engine ---
engine = PersonalizedSearchEngine(
    reviews_df=df_reviews,
    behaviors_df=df_behaviors,
    users_df=df_users,
    groq_api_key="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"
)

# --- 3. Run a Personalized Search ---
TARGET_USER_ID = "USER-00001" # Ensure this ID exists in your CSV
SEARCH_QUERY = "gaming laptop" 

print(f"\n🔎 Searching for '{SEARCH_QUERY}' for User {TARGET_USER_ID}...")

# Execute the search
results = engine.personalized_search(
    user_id=TARGET_USER_ID,
    query=SEARCH_QUERY,
    search_type="text", 
    limit=5              
)

# --- 4. Display Results ---
# FIX: We only pass 'results' because the AI explanation is already inside them.
display_personalized_results(results)

📂 Loading your data files...
✅ Data loaded successfully!

🔎 Searching for 'gaming laptop' for User USER-00001...

🔎 SEARCHING (text): 'gaming laptop' for User USER-00001...
   💰 Max Budget Set To: $10000.0
   🤖 Generating Smart AI explanations...

✨ Top 5 Personalized Recommendations:

1. 📦 ASUS VivoBook
   💰 Price: $2672.78
   📂 Category: Laptops
   ⭐ Relevance Score: 0.8604
   💡 Why this fits: "The ASUS VivoBook is a great choice for a gaming laptop due to its powerful specs, which, despite the relatively high price of $2672.78, offer excellent value and relevance to gaming needs, making it a worthwhile investment for those seeking a high-performance laptop."
------------------------------------------------------------

2. 📦 Apple MacBook Pro 14
   💰 Price: $2951.37
   📂 Category: Laptops
   ⭐ Relevance Score: 0.8413
   💡 Why this fits: "The Apple MacBook Pro 14 is a great choice for a gaming laptop due to its powerful M1 Pro or M1 Max chip, stunning 14-inch Liquid Retina XDR display

In [3]:
# --- 3. Run a Personalized IMAGE Search ---
TARGET_USER_ID = "USER-00001" # Ensure this user exists

# 1. Set the path to your image file (Upload this file to Jupyter first!)
IMAGE_FILE_PATH = "blueshirt.jpg"  # <--- REPLACE with your actual image filename

print(f"\n🔎 Searching using IMAGE '{IMAGE_FILE_PATH}' for User {TARGET_USER_ID}...")

try:
    # Execute the search
    results = engine.personalized_search(
        user_id=TARGET_USER_ID,
        query=IMAGE_FILE_PATH,   # <--- Pass the FILE PATH here
        search_type="image",     # <--- Change this to "image"
        limit=5              
    )

    # --- 4. Display Results ---
    display_personalized_results(results)

except FileNotFoundError:
    print(f"❌ Error: The file '{IMAGE_FILE_PATH}' was not found.")
    print("👉 Please upload an image file to the notebook folder and update the filename.")
except Exception as e:
    print(f"❌ An error occurred: {e}")


🔎 Searching using IMAGE 'blueshirt.jpg' for User USER-00001...

🔎 SEARCHING (image): 'blueshirt.jpg' for User USER-00001...
   💰 Max Budget Set To: $10000.0
   🤖 Generating Smart AI explanations...

✨ Top 5 Personalized Recommendations:

1. 📦 Uniqlo Supima Cotton
   💰 Price: $27.72
   📂 Category: T-Shirts
   ⭐ Relevance Score: 0.3979
   💡 Why this fits: "The Uniqlo Supima Cotton product is a great choice for your search because it offers high-quality, extra-long staple cotton fabric at an affordable price of $27.72, making it an excellent value for those seeking a durable and comfortable cotton product."
------------------------------------------------------------

2. 📦 Uniqlo AIRism
   💰 Price: $24.01
   📂 Category: T-Shirts
   ⭐ Relevance Score: 0.3970
   💡 Why this fits: "The Uniqlo AIRism product is a great choice for your search because it offers a highly breathable, moisture-wicking, and quick-drying fabric technology at an affordable price of $24.01, making it an excellent valu

# recommendation engine

In [1]:
import pandas as pd
import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, Range
from sentence_transformers import SentenceTransformer
from groq import Groq

# --- CONFIGURATION ---
client = QdrantClient(path="qdrant_local_data")
model = SentenceTransformer('clip-ViT-B-32') # Must match your product embeddings
COLLECTION_PRODUCTS = "products_db"
COLLECTION_USERS = "users_collection"

class UserDiscoveryEngine:
    def __init__(self, users_df, groq_api_key):
        self.users_df = users_df
        self.groq_client = Groq(api_key=groq_api_key)
        self.users_df = self._clean_df(self.users_df)

    def _clean_df(self, df):
        """Helper to fix the semicolon CSV issue if present."""
        if len(df.columns) == 1 and ';' in df.columns[0]:
            col_name = df.columns[0]
            split_data = df[col_name].str.split(';', expand=True)
            split_data.columns = col_name.split(';')
            return split_data
        return df

    def _get_user_safe_id(self, user_id_str):
        """Standardizes the ID for lookup."""
        import uuid
        return str(uuid.uuid5(uuid.NAMESPACE_DNS, str(user_id_str).strip()))

    def _generate_feed_explanation(self, product_name, price, user_interests):
        """
        Explains why a product fits a user's general profile.
        """
        try:
            prompt = (
                f"A user is interested in '{user_interests}'. "
                f"Explain why recommending '{product_name}' (${price}) is a smart choice for them. "
                "Don't say 'based on your interest', just give a punchy 1-sentence sales pitch."
            )
            chat = self.groq_client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model="llama-3.3-70b-versatile",
                temperature=0.6,
                max_tokens=60
            )
            return chat.choices[0].message.content.strip()
        except:
            return "Matches your personal taste profile."

    def suggest_products(self, user_id_str, limit=5):
        print(f"\n👤 Generating Discovery Feed for User: {user_id_str}...")
        
        # 1. Get User's Vector from DB
        hashed_id = self._get_user_safe_id(user_id_str)
        try:
            user_data = client.retrieve(
                collection_name=COLLECTION_USERS,
                ids=[hashed_id],
                with_vectors=True
            )
            
            if not user_data:
                print("⚠️ User vector not found. Creating temporary profile based on CSV...")
                # Fallback: Create vector on the fly from CSV text
                user_row = self.users_df[self.users_df['user_id'] == user_id_str].iloc[0]
                interests = str(user_row.get('preferred_sectors', 'General'))
                budget = float(user_row.get('monthly_limit', 10000))
                user_vector = model.encode(interests).tolist()
            else:
                # Use the pre-calculated vector from the DB
                user_vector = user_data[0].vector['preference_vector']
                user_row = self.users_df[self.users_df['user_id'] == user_id_str].iloc[0]
                interests = str(user_row.get('preferred_sectors', 'General'))
                budget = float(user_row.get('monthly_limit', 10000))

        except Exception as e:
            print(f"❌ Error retrieving user: {e}")
            return []

        print(f"   ❤️  Known Interests: {interests}")
        print(f"   💰 Budget Cap: ${budget}")

        # 2. Search Products using User Vector (The "Feed" Logic)
        # We are finding products that are mathematically similar to the USER
        price_filter = Filter(
            must=[FieldCondition(key="price", range=Range(lte=budget))]
        )

        try:
            hits = client.query_points(
                collection_name=COLLECTION_PRODUCTS,
                query=user_vector,   # <--- Searching with USER vector, not text query
                using="text",        # Aligning with product text space
                query_filter=price_filter,
                limit=limit,
                with_payload=True
            )
        except:
            # Fallback if 'text' vector name differs
            hits = client.query_points(
                collection_name=COLLECTION_PRODUCTS,
                query=user_vector,
                using="image",
                query_filter=price_filter,
                limit=limit,
                with_payload=True
            )

        # 3. Format & Explain
        feed = []
        print("   🤖 AI Analyzing matches...")
        
        for point in hits.points:
            p_data = point.payload
            
            explanation = self._generate_feed_explanation(
                p_data.get('name'), 
                p_data.get('price'), 
                interests
            )

            feed.append({
                "name": p_data.get('name'),
                "price": p_data.get('price'),
                "category": p_data.get('category'),
                "score": point.score,
                "why": explanation
            })

        return feed

# --- DISPLAY FUNCTION ---
def display_feed(feed):
    if not feed:
        print("❌ No recommendations generated.")
        return
        
    print("\n✨ YOUR PERSONAL DAILY FEED ✨")
    print("="*60)
    for i, item in enumerate(feed, 1):
        print(f"{i}. {item['name']}")
        print(f"   💵 ${item['price']:.2f}")
        print(f"   💡 {item['why']}")
        print(f"   📊 Match Score: {item['score']:.4f}")
        print("-" * 40)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [2]:
# 1. Setup
GROQ_API_KEY = "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"
df_users = pd.read_csv("users.csv", sep=";")

# 2. Init Engine
feed_engine = UserDiscoveryEngine(df_users, GROQ_API_KEY)

# 3. Pick a user and Generate Feed
target_user = "USER-00001" # Replace with real ID
my_feed = feed_engine.suggest_products(target_user, limit=4)

# 4. Show
display_feed(my_feed)


👤 Generating Discovery Feed for User: USER-00001...
   ❤️  Known Interests: Wine|Healthcare|Food|Furniture
   💰 Budget Cap: $10000.0
   🤖 AI Analyzing matches...

✨ YOUR PERSONAL DAILY FEED ✨
1. Braun ThermoScan 7
   💵 $175.21
   💡 The Braun ThermoScan 7 is a smart investment for anyone who values health and wellness, offering fast and accurate temperature readings to help you take care of yourself and your loved ones.
   📊 Match Score: 0.6727
----------------------------------------
2. Braun ThermoScan 7
   💵 $339.13
   💡 The Braun ThermoScan 7 is a smart investment for anyone who values wellness and self-care, offering fast and accurate temperature readings to help you take control of your health, perfectly complementing a lifestyle that prioritizes good food, fine wine, and a comfortable home.
   📊 Match Score: 0.6727
----------------------------------------
3. Braun ThermoScan 7
   💵 $50.33
   💡 The Braun ThermoScan 7 is a savvy investment for anyone who values wellness, offering 